### TRANSFERIR LOS DATOS A LA CAPA SILVER
**IMPORTAMOS LAS LIBRERIAS**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

**EXTRAEMOS LOS DATOS DE LA CAPA BRONZE**

In [0]:
silver_bronze= spark.table("proyecto_spotify.bronze.spotify_tracks")
#display(silver_bronze.limit(5))

**LIMPIEZA DE DATOS**

In [0]:
window = Window.partitionBy("track_id").orderBy(col("extraction_timestamp").desc())

silver_clean = (
    silver_bronze
    .withColumn(
        "rn",
        row_number().over(window)
    )
    .filter(col("rn") == 1)
    .drop("rn")
)

In [0]:
silver_clean = silver_clean.filter(col("track_id").isNotNull())
silver_clean = silver_clean.filter(col("track_name").isNotNull() & (trim(col("track_name")) != ""))

In [0]:
silver_clean = silver_clean.filter(col("popularity") > 0)

In [0]:
silver_clean = silver_clean.filter(col("duration_ms") >= 60000)

**TRANSFORMACIÓN DE DATOS**

In [0]:
silver_clean = silver_clean.withColumn("duration_minutes",round(col("duration_ms") / 60000, 2))

In [0]:
silver_clean = silver_clean.withColumn(
    "release_date_precision",
    when(length(col("release_date")) == 4, "year")
    .when(length(col("release_date")) == 7, "month")
    .when(length(col("release_date")) == 10, "day")
    .otherwise("unknown")
)

silver_clean = silver_clean.withColumn(
    "release_date_clean",
    when(
        length(col("release_date")) == 10,
        to_date(col("release_date"), "yyyy-MM-dd")
    )
    .when(
        length(col("release_date")) == 7,
        to_date(
            concat(col("release_date"), lit("-01")),
            "yyyy-MM-dd"
        )
    )
    .when(
        length(col("release_date")) == 4,
        to_date(
            concat(col("release_date"), lit("-01-01")),
            "yyyy-MM-dd"
        )
    )
    .otherwise(None)
)

In [0]:
silver_clean = silver_clean.filter((col("popularity")>=0) & (col("popularity")<=100)
                ).withColumn(
                    "popularity_level",  
                    when(col("popularity") <= 25, "Baja")
                    .when(col("popularity") <= 50, "Media")
                    .when(col("popularity") <= 75, "Alta")
                    .when(col("popularity") <= 100, "Muy alta")
                    .otherwise("Desconocida")
                )

In [0]:
print("Cantidad de Registros en Capa Bronze: ", silver_bronze.count())
print("Cantidad de Registros en Capa Silver: ", silver_clean.count())

In [0]:
display(silver_clean.limit(10))

**GUARDAMOS LOS DATOS EN LA CAPA SILVER**

In [0]:
(
    silver_clean
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "proyecto_spotify.silver.spotify_tracks"
    )
)